[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-2-COMPLETE_executed.ipynb)

---


# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 2: List Scheduling Algorithm

### Goal and inputs

This notebook implements a greedy list-scheduling heuristic for rearranging containers in a bay. Each container has an export priority, a stack, and a tier. A lower priority number means that the container should be accessible earlier. The algorithm repeatedly scores feasible moves and executes the highest-scoring move.

The inputs are the initial container positions, the number of stacks, the tier capacity, and a maximum number of moves. The outputs are the final positions, the move sequence, an execution log, and visualizations of the bay. The method is a heuristic: it does not guarantee a globally optimal arrangement or that all priority violations can be eliminated within the move limit.

### Scope and modeling assumptions

- Stack and tier indices start at zero. Tier 0 is the bottom position; a larger tier index is higher in the stack.
- Containers occupy contiguous positions from tier 0 upward. Only the top container in a stack can be moved, and a move places it on top of another stack with available capacity.
- A priority violation is an adjacent pair in one stack where the lower container has a larger priority number than the container immediately above it. This is the violation measure used by both the score and the stopping condition.
- The model does not include travel time, crane interference, container dimensions, or other operational constraints. The move-efficiency term is a scoring preference, not a physical travel-time model.

### Scoring and selection

A candidate list is generated from the accessible top containers and feasible destination stacks. The algorithm selects the candidate with the largest composite score. It does not use a heap-based priority queue, and it does not require a separate optimization solver.

For a configuration \(x\), let \(V(x)\) be the number of adjacent priority violations, \(h_s(x)\) the height of stack \(s\), and  $$B(x) = \sum_{s=0}^{S-1} \left|h_s(x) - \bar{h}(x)\right|,\qquad\bar{h}(x) = \frac{1}{S}\sum_{s=0}^{S-1} h_s(x).$$
For a feasible move \(m\), the score is
$$\operatorname{score}(m) = 50\bigl(V(x)-V(x_m)\bigr)+ 20\bigl(B(x)-B(x_m)\bigr)+ \max\left(0, 15 - 2d(m) - r(m)\right)+ 15\frac{10-p(m)}{10}.$$
Here \(x_m\) is the configuration after the move, \(d(m)\) is the absolute difference between the source and destination stack indices, \(r(m)\) is the source tier index, and \(p(m)\) is the moved container's priority. Thus, a positive violation or balance difference represents an immediate reduction in that quantity; a negative difference represents an immediate increase. The first component is calculated by evaluating the actual configuration after the move. The score also includes an efficiency preference and a priority bonus, so a selected move is not guaranteed to reduce violations on every iteration.

### Algorithm and termination

At each iteration, the method checks the violation count, generates feasible moves, scores them, and executes the highest-scoring candidate. It stops when there are no violations, when no feasible moves exist, or when the move limit is reached. Since the selected move need not reduce the violation count, the move limit also bounds the number of iterations.

For \(n\) containers, \(S\) stacks, \(T\) tiers, and a move limit \(M\), this implementation takes at most \(M\) iterations. A direct upper bound for the work is \(O\bigl(MS^2(n+ST)\bigr)\): there are at most \(S(S-1)\) candidate moves per iteration, and scoring a candidate evaluates stack heights and violations. This is a bound for the implemented heuristic, not a guarantee that it will find a feasible priority arrangement.

### Concrete example

The bay has three stacks and four available tiers per stack:

- Stack 0: `[E (priority 5), B (priority 2)]`, with `E` above `B`.
- Stack 1: `[A (priority 1), D (priority 4)]`, with `A` above `D`.
- Stack 2: `[C (priority 3)]`.

The bottom-to-top order in each list is intentional. The adjacent pair in Stack 1 is a priority violation because the lower container `D` has a larger priority number than the container `A` above it.

In [ ]:
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

plt.style.use("default")
sns.set_palette("husl")


@dataclass
class Container:
    """A container with an export priority and a current bay position."""

    id: str
    priority: int
    current_stack: int
    current_tier: int


@dataclass
class Move:
    """A feasible relocation of one container between stacks."""

    container_id: str
    from_stack: int
    from_tier: int
    to_stack: int
    to_tier: int
    score: float


class ListScheduler:
    """Greedy list scheduling for the yard pre-marshalling problem."""

    def __init__(self, stacks: int, tiers: int):
        if stacks <= 0 or tiers <= 0:
            raise ValueError("The bay must have at least one stack and one tier.")
        self.stacks = stacks
        self.tiers = tiers
        self.move_history: List[Move] = []
        self.execution_log: List[str] = []

    def get_stack_height(self, containers: List[Container], stack: int) -> int:
        """Return the number of containers currently in a stack."""
        return sum(container.current_stack == stack for container in containers)

    def get_top_container(
        self, containers: List[Container], stack: int
    ) -> Optional[Container]:
        """Return the highest-tier container, which is the only movable one."""
        stack_containers = [
            container
            for container in containers
            if container.current_stack == stack
        ]
        if not stack_containers:
            return None
        return max(stack_containers, key=lambda container: container.current_tier)

    def count_priority_violations(self, containers: List[Container]) -> int:
        """Count adjacent bottom-to-top priority inversions in all stacks."""
        grid = [[None for _ in range(self.tiers)] for _ in range(self.stacks)]
        for container in containers:
            grid[container.current_stack][container.current_tier] = container.priority

        violations = 0
        for stack in range(self.stacks):
            for tier in range(self.tiers - 1):
                lower_priority = grid[stack][tier]
                upper_priority = grid[stack][tier + 1]
                if (
                    lower_priority is not None
                    and upper_priority is not None
                    and lower_priority > upper_priority
                ):
                    violations += 1
        return violations

    def calculate_move_score(self, containers: List[Container], move: Move) -> float:
        """Score a candidate using its immediate effects and fixed preferences."""
        container = next(
            item for item in containers if item.id == move.container_id
        )

        violations_before = self.count_priority_violations(containers)
        hypothetical_containers = [
            Container(
                id=item.id,
                priority=item.priority,
                current_stack=(
                    move.to_stack if item.id == move.container_id else item.current_stack
                ),
                current_tier=(
                    move.to_tier if item.id == move.container_id else item.current_tier
                ),
            )
            for item in containers
        ]
        violations_after = self.count_priority_violations(hypothetical_containers)
        violation_reduction = violations_before - violations_after
        score = 50 * violation_reduction

        heights = [
            self.get_stack_height(containers, stack)
            for stack in range(self.stacks)
        ]
        average_height = sum(heights) / self.stacks
        balance_before = sum(abs(height - average_height) for height in heights)

        updated_heights = heights.copy()
        updated_heights[move.from_stack] -= 1
        updated_heights[move.to_stack] += 1
        balance_after = sum(
            abs(height - average_height) for height in updated_heights
        )
        balance_improvement = balance_before - balance_after
        score += 20 * balance_improvement

        distance = abs(move.from_stack - move.to_stack)
        tier_index_penalty = move.from_tier
        efficiency = 15 - (2 * distance) - tier_index_penalty
        score += max(0, efficiency)

        priority_bonus = 15 * (10 - container.priority) / 10
        score += priority_bonus
        return score

    def generate_valid_moves(self, containers: List[Container]) -> List[Move]:
        """List all top-container moves to other stacks with free capacity."""
        moves = []
        for stack in range(self.stacks):
            top_container = self.get_top_container(containers, stack)
            if top_container is None:
                continue

            for target_stack in range(self.stacks):
                if target_stack == stack:
                    continue

                target_height = self.get_stack_height(containers, target_stack)
                if target_height >= self.tiers:
                    continue

                move = Move(
                    container_id=top_container.id,
                    from_stack=stack,
                    from_tier=top_container.current_tier,
                    to_stack=target_stack,
                    to_tier=target_height,
                    score=0.0,
                )
                move.score = self.calculate_move_score(containers, move)
                moves.append(move)
        return moves

    def execute_move(self, containers: List[Container], move: Move) -> List[Container]:
        """Apply a feasible top-container move without mutating the input list."""
        moved_container = next(
            (item for item in containers if item.id == move.container_id), None
        )
        if moved_container is None:
            raise ValueError(f"Unknown container ID: {move.container_id}")

        top_container = self.get_top_container(containers, move.from_stack)
        target_height = self.get_stack_height(containers, move.to_stack)
        if top_container is None or top_container.id != move.container_id:
            raise ValueError("Only the current top container can be moved.")
        if moved_container.current_tier != move.from_tier:
            raise ValueError("The move's source tier does not match the current state.")
        if move.to_stack == move.from_stack:
            raise ValueError("The source and destination stacks must differ.")
        if target_height >= self.tiers or move.to_tier != target_height:
            raise ValueError("The destination must be the current top free position.")

        return [
            Container(
                id=item.id,
                priority=item.priority,
                current_stack=(
                    move.to_stack if item.id == move.container_id else item.current_stack
                ),
                current_tier=(
                    move.to_tier if item.id == move.container_id else item.current_tier
                ),
            )
            for item in containers
        ]

    def solve(
        self, initial_containers: List[Container], max_moves: int = 100
    ) -> Tuple[List[Container], List[Move]]:
        """Run greedy list scheduling until a stopping condition is reached."""
        if max_moves < 0:
            raise ValueError("max_moves must be non-negative.")

        containers = [
            Container(
                item.id, item.priority, item.current_stack, item.current_tier
            )
            for item in initial_containers
        ]
        validate_configuration(containers, self.stacks, self.tiers)
        self.move_history = []
        self.execution_log = []

        iteration = 0
        while iteration < max_moves:
            violations = self.count_priority_violations(containers)
            if violations == 0:
                self.execution_log.append(
                    f"Iteration {iteration}: No adjacent priority violations remain."
                )
                break

            valid_moves = self.generate_valid_moves(containers)
            if not valid_moves:
                self.execution_log.append(
                    f"Iteration {iteration}: No feasible moves are available."
                )
                break

            best_move = max(valid_moves, key=lambda candidate: candidate.score)
            containers = self.execute_move(containers, best_move)
            self.move_history.append(best_move)

            new_violations = self.count_priority_violations(containers)
            self.execution_log.append(
                f"Iteration {iteration}: Move {best_move.container_id} from stack "
                f"{best_move.from_stack} to stack {best_move.to_stack}, "
                f"score={best_move.score:.1f}, adjacent violations: "
                f"{violations} -> {new_violations}"
            )
            iteration += 1
        else:
            if self.count_priority_violations(containers) > 0:
                self.execution_log.append(
                    f"Move limit reached after {max_moves} moves."
                )

        return containers, self.move_history


In [ ]:
def validate_configuration(
    containers: List[Container], stacks: int, tiers: int
) -> bool:
    """Check container identity, capacity, and contiguous stack positions."""
    identifiers = [container.id for container in containers]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Container IDs must be unique.")

    positions = set()
    stack_tiers = [[] for _ in range(stacks)]
    for container in containers:
        if not 0 <= container.current_stack < stacks:
            raise ValueError(f"Invalid stack index for container {container.id}.")
        if not 0 <= container.current_tier < tiers:
            raise ValueError(f"Invalid tier index for container {container.id}.")

        position = (container.current_stack, container.current_tier)
        if position in positions:
            raise ValueError("Two containers cannot occupy the same position.")
        positions.add(position)
        stack_tiers[container.current_stack].append(container.current_tier)

    for tier_indices in stack_tiers:
        if sorted(tier_indices) != list(range(len(tier_indices))):
            raise ValueError("Containers in each stack must occupy contiguous tiers.")
    return True

In [ ]:
def validate_move_history(
    scheduler: ListScheduler,
    initial_containers: List[Container],
    final_containers: List[Container],
    move_history: List[Move],
    stacks: int,
    tiers: int,
) -> bool:
    """Replay the recorded moves and verify that they reproduce the final state."""
    replayed = [
        Container(item.id, item.priority, item.current_stack, item.current_tier)
        for item in initial_containers
    ]

    for move in move_history:
        replayed = scheduler.execute_move(replayed, move)
        validate_configuration(replayed, stacks, tiers)

    final_positions = {
        item.id: (item.priority, item.current_stack, item.current_tier)
        for item in final_containers
    }
    replayed_positions = {
        item.id: (item.priority, item.current_stack, item.current_tier)
        for item in replayed
    }
    if replayed_positions != final_positions:
        raise AssertionError("Replaying the move history does not reproduce the final state.")
    return True

In [ ]:
def visualize_bay_state(
    containers: List[Container],
    stacks: int,
    tiers: int,
    title: str,
    ax=None,
):
    """Plot stack positions and encode export priority with a color scale."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container

    priorities = [container.priority for container in containers]
    minimum_priority = min(priorities, default=0)
    maximum_priority = max(priorities, default=1)
    if minimum_priority == maximum_priority:
        maximum_priority += 1
    normalizer = Normalize(vmin=minimum_priority, vmax=maximum_priority)
    color_map = plt.cm.RdYlBu_r

    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            x_position = stack * 1.2
            y_position = tier * 0.8
            if container is not None:
                rectangle = plt.Rectangle(
                    (x_position, y_position),
                    1.0,
                    0.6,
                    facecolor=color_map(normalizer(container.priority)),
                    edgecolor="black",
                    linewidth=1.5,
                )
                ax.add_patch(rectangle)
                ax.text(
                    x_position + 0.5,
                    y_position + 0.3,
                    f"{container.id}\n(Priority {container.priority})",
                    ha="center",
                    va="center",
                    fontweight="bold",
                    fontsize=9,
                )
            else:
                rectangle = plt.Rectangle(
                    (x_position, y_position),
                    1.0,
                    0.6,
                    facecolor="lightgray",
                    edgecolor="gray",
                    linewidth=1,
                    alpha=0.4,
                )
                ax.add_patch(rectangle)

    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xticks([stack * 1.2 + 0.5 for stack in range(stacks)])
    ax.set_xticklabels([f"S{stack}" for stack in range(stacks)])
    ax.set_yticks([tier * 0.8 + 0.3 for tier in range(tiers)])
    ax.set_yticklabels([f"T{tier}" for tier in range(tiers)])
    ax.set_xlabel("Stack (left to right)")
    ax.set_ylabel("Tier (bottom to top)")
    ax.set_title(title, fontweight="bold")
    ax.set_aspect("equal")
    ax.grid(False)

    color_scale = ScalarMappable(norm=normalizer, cmap=color_map)
    color_scale.set_array([])
    colorbar = ax.figure.colorbar(color_scale, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label("Export priority (lower number = earlier)")


def visualize_algorithm_progression(
    initial_containers: List[Container],
    final_containers: List[Container],
    move_history: List[Move],
    stacks: int,
    tiers: int,
):
    """Plot the initial state, selected intermediate states, and final state."""
    n_moves = len(move_history)
    if n_moves == 0:
        print("No moves were executed; the initial state is also the final state.")
        return

    n_stages = min(5, n_moves + 1)
    figure, axes = plt.subplots(1, n_stages, figsize=(5 * n_stages, 5))
    if n_stages == 1:
        axes = [axes]

    visualize_bay_state(
        initial_containers, stacks, tiers, "Initial state", axes[0]
    )

    current_containers = [
        Container(item.id, item.priority, item.current_stack, item.current_tier)
        for item in initial_containers
    ]
    previous_move_index = -1

    for stage in range(1, n_stages):
        if n_stages == 2:
            move_index = n_moves - 1
        else:
            move_index = round(stage * (n_moves - 1) / (n_stages - 1))

        for index in range(previous_move_index + 1, move_index + 1):
            move = move_history[index]
            moved = next(
                item for item in current_containers if item.id == move.container_id
            )
            moved.current_stack = move.to_stack
            moved.current_tier = move.to_tier
        previous_move_index = move_index

        if stage == n_stages - 1:
            stage_containers = final_containers
            title = f"Final state ({n_moves} moves)"
        else:
            stage_containers = current_containers
            title = f"After {move_index + 1} moves"

        visualize_bay_state(
            stage_containers, stacks, tiers, title, axes[stage]
        )

    figure.tight_layout()
    plt.show()

In [ ]:
print("=" * 60)
print("YARD PRE-MARSHALLING PROBLEM - LIST SCHEDULING")
print("=" * 60)

containers = [
    Container("E", 5, 0, 1),
    Container("B", 2, 0, 0),
    Container("A", 1, 1, 1),
    Container("D", 4, 1, 0),
    Container("C", 3, 2, 0),
]

stacks, tiers = 3, 4
validate_configuration(containers, stacks, tiers)

print("\nProblem configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers per stack: {tiers}")
print(f"- Containers: {len(containers)}")

print("\nInitial container positions:")
for container in containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

initial_violations = ListScheduler(stacks, tiers).count_priority_violations(
    containers
)
print(f"\nInitial adjacent priority violations: {initial_violations}")

visualize_bay_state(
    containers, stacks, tiers, "Initial bay configuration"
)

In [ ]:
scheduler = ListScheduler(stacks, tiers)

print("\n" + "=" * 60)
print("LIST SCHEDULING EXECUTION")
print("=" * 60)

start_time = time.time()
final_containers, move_history = scheduler.solve(containers, max_moves=50)
execution_time = time.time() - start_time

print(f"\nExecution time: {execution_time:.4f} seconds")
print(f"Moves executed: {len(move_history)}")

print("\nExecution log:")
for log_entry in scheduler.execution_log:
    print(f"  {log_entry}")

In [ ]:
print("\n" + "=" * 60)
print("FINAL SOLUTION ANALYSIS")
print("=" * 60)

validate_configuration(final_containers, stacks, tiers)
validate_move_history(
    scheduler,
    containers,
    final_containers,
    move_history,
    stacks,
    tiers,
)

print("\nFinal container positions:")
for container in final_containers:
    print(
        f"  Container {container.id}: priority {container.priority}, "
        f"stack {container.current_stack}, tier {container.current_tier}"
    )

final_violations = scheduler.count_priority_violations(final_containers)
print("\nSolution assessment:")
print(f"  Initial adjacent priority violations: {initial_violations}")
print(f"  Final adjacent priority violations: {final_violations}")

if final_violations == 0:
    print("  No adjacent priority violations remain.")
else:
    print(
        f"  The final configuration contains {final_violations} "
        "adjacent priority violation(s)."
    )

visualize_algorithm_progression(
    containers, final_containers, move_history, stacks, tiers
)

## Interpreting the heuristic

The move score combines four preferences: immediate change in adjacent priority violations, change in stack-height balance, a distance and source-tier preference, and a bonus for containers with smaller priority numbers. The weights determine how strongly each component influences the greedy choice. They are part of this example's scoring rule, not a proof that the chosen move is best for the complete sequence.

A zero final violation count means that every adjacent pair in every stack satisfies the priority rule defined in this notebook. It does not establish global optimality with respect to the number of moves or any unmodeled operational objective. If violations remain, inspect the execution log to see whether the move limit was reached or no feasible move was available. The heuristic can also choose a move that temporarily increases the violation count because its score includes several components.

The move history and the final configuration can be checked independently by replaying every move from the initial state. Configuration checks also ensure that container IDs are unique, positions are within the bay, and tiers in each stack remain contiguous. These checks verify the implementation's state transitions; they do not turn the heuristic into an optimality method.

### Strengths and limitations

**Strengths**

- The candidate moves and their score components are visible, making each greedy decision inspectable.
- Top-container accessibility and stack capacity are enforced when moves are generated and executed.
- A finite move limit bounds the number of iterations.

**Limitations**

- Greedy selection has no global optimality guarantee and may fail to eliminate all violations.
- Results depend on the scoring weights and on the instance's initial arrangement.
- The implementation does not detect repeated configurations; the move limit bounds any cycling behavior.
- The model omits operational details such as equipment conflicts and travel-time costs.